In [37]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from ase import Atoms
from ase.io import read, write

In [ ]:
input_file = "Fe_E111_110_naked.lmp"
output_file = "Fe_E111_110_slabbed.lmp"

num_y_layers = 70
num_fixed_layers = 1

input_dir = Path(f"../input").resolve()
input_path = input_dir / input_file

atoms = read(input_path, format="lammps-data")

In [39]:
# -------------------------
# 1. Periodicity and Vacuum
# -------------------------
atoms.set_pbc([True, False, True])
atoms.center()

# -------------------------
# 2. Y-coordinates
# -------------------------
y = atoms.positions[:, 1]
y_min, y_max = y.min(), y.max()

# -------------------------
# 3. Determine layer index
# -------------------------
layer_thickness = (y_max - y_min) / num_y_layers

# Compute integer layer index for each atom
layer_index = ((y - y_min) / layer_thickness).astype(int)

# Safety clamp (avoid rare floating-point overflow on top layer)
layer_index = np.clip(layer_index, 0, num_y_layers - 1)

# -------------------------
# 4. Assign LAMMPS types
# -------------------------
types = np.ones(len(atoms), dtype=int)

fixed_mask = (
    (layer_index < num_fixed_layers) |
    (layer_index >= num_y_layers - num_fixed_layers)
)

types[fixed_mask] = 2

atoms.set_array("type", types)

# -------------------------
# 5. Write LAMMPS data file
# -------------------------
write(
    input_dir / output_file,
    atoms,
    format="lammps-data",
    atom_style="atomic"
)